# 1D Burgers' Equation with FNO1D

Learn the solution operator of the viscous Burgers' equation with a 1D Fourier Neural Operator
([Li et al., 2021](https://arxiv.org/abs/2010.08895)). The operator maps an initial velocity profile
$u_0(x)$ to the solution $u(x, T)$ of

$$u_t + u\, u_x = \nu\, u_{xx}, \quad x \in [0, 1) \text{ periodic}.$$

Burgers' equation is a standard nonlinear benchmark: its solutions steepen into shock-like fronts
that viscosity then smooths out.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`NU` is the viscosity and `T_END` the time the operator maps to.

In [2]:
N_SAMPLES = 64
NX = 32
T_END = 0.5
NU = 0.02
MODES = 8
WIDTH = 32
DEPTH = 3
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01

## Load the data

`BurgersDataset` samples random initial conditions and integrates them to `T_END` with a
pseudospectral solver.

In [3]:
import keras

from keras_sciml.datasets import BurgersDataset
from keras_sciml.losses import H1Loss, RelativeL2Loss
from keras_sciml.models import FNO1D

keras.utils.set_random_seed(42)

ds = BurgersDataset(mode="pseudospectral", n_samples=N_SAMPLES, nx=NX, t_end=T_END, nu=NU, seed=42)
u0, ut = ds.as_tuple()

n_train = int(0.8 * N_SAMPLES)
u0_train, ut_train = u0[:n_train], ut[:n_train]
u0_test, ut_test = u0[n_train:], ut[n_train:]
print(f"Train: {u0_train.shape}, Test: {u0_test.shape}")

Train: (51, 32, 1), Test: (13, 32, 1)


## Define the model

The model is trained with an $H^1$ Sobolev loss, which penalises errors in the spatial derivative as
well as in the values. This helps the model get the steep fronts right.

In [4]:
model = FNO1D(
    modes=MODES,
    width=WIDTH,
    depth=DEPTH,
    out_channels=1,
    activation="gelu",
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=H1Loss(alpha=0.05, dx=1.0 / NX),
    metrics=[RelativeL2Loss(name="rel_l2")],
)

## Train

In [5]:
history = model.fit(
    u0_train,
    ut_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(u0_test, ut_test),
    verbose=2,
)

Epoch 1/30


4/4 - 3s - 872ms/step - loss: 0.2853 - rel_l2: 0.6743 - val_loss: 0.1034 - val_rel_l2: 0.4748


Epoch 2/30


4/4 - 0s - 21ms/step - loss: 0.1009 - rel_l2: 0.3554 - val_loss: 0.0586 - val_rel_l2: 0.4059


Epoch 3/30


...

4/4 - 0s - 20ms/step - loss: 8.4952e-04 - rel_l2: 0.0351 - val_loss: 0.0017 - val_rel_l2: 0.0568


Epoch 29/30


4/4 - 0s - 22ms/step - loss: 8.3182e-04 - rel_l2: 0.0352 - val_loss: 0.0017 - val_rel_l2: 0.0566


Epoch 30/30


4/4 - 0s - 20ms/step - loss: 8.2361e-04 - rel_l2: 0.0357 - val_loss: 0.0017 - val_rel_l2: 0.0569


## Evaluate

In [6]:
test_loss = float(model.evaluate(u0_test, ut_test, verbose=0)[0])
print(f"Test H1 loss: {test_loss:.4f}")

Test H1 loss: 0.0017
